In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pygmt
import pandas as pd
import geopandas as gpd
import glob
from shapely.geometry import Point, box
import os

# 資料預處理
## 計算歷年雙北市測速照相密度

In [2]:
years =[110,111,112,113] 
CITY = ['臺北市', '新北市']
CITY_ENG = {'臺北市':'Taipei','新北市':'New_Taipei'}
for k in CITY:
    for i in years:
        filename = f'./stations/NPA_TD1_{i}.csv'
        df = pd.read_csv(filename, sep= ',', header = 0, skipinitialspace=True)
        df = pd.DataFrame(df[1:])
        df.reset_index(inplace=True, drop = True)
        df['limit'] = df['limit'].astype('int').copy()
        
        path = glob.glob('./data/*.shp')
        
        gdf = gpd.read_file(path[1], encoding='utf-8')
        
        if gdf.crs.to_string() != "EPSG:4326":
            gdf = gdf.to_crs("EPSG:4326")
        
        gdf =gdf.sort_values(by='TOWNID')
        gdf.reset_index(drop = True,inplace = True)
        
        COUNTY = gdf[gdf['COUNTYNAME'] == k].copy()
        
        df_count = gpd.GeoDataFrame(df.copy(),
                                    geometry=gpd.points_from_xy(df.copy().Longitude,
                                                                df.copy().Latitude),
                                    crs="EPSG:4326" # 指定這是經緯度
        )
        joined = gpd.sjoin(COUNTY, df_count, how="left", predicate="contains") #by gemini 3
        # 假設 joined_gdf 是你做完 sjoin 的結果
        # 'CTYNAME' 是你的行政區名稱欄位
        counts = joined.groupby(joined.index)['index_right'].count()
        COUNTY['point_count'] = counts
        
        COUNTY['CBC'] = COUNTY.groupby(COUNTY.COUNTYID)['point_count'].transform('sum').copy()
        
        county_outlines = gdf.dissolve(gdf.COUNTYNAME)['geometry']
        COUNTY_outlines = gpd.GeoDataFrame(
            geometry=[county_outlines[k]], 
            crs=gdf.crs
        )
        
        area = pd.read_csv('area.csv', sep = ',')
        # town_area = area.set_index('TOWNNAME', drop = True)
        total_area = area.groupby(area.COUNTYNAME)['AREA'].sum()
        total_area.reindex(county_outlines.index)
        
        COUNTY['TOTAL_AREA'] = COUNTY['COUNTYNAME'].map(total_area)
        if 'TOWN_AREA'  in COUNTY.columns:
            COUNTY = COUNTY.drop(columns=['TOWN_AREA'])
            COUNTY = COUNTY.merge(area[['TOWNNAME','AREA']], on=['TOWNNAME'], how = 'left')
        else:
            COUNTY = COUNTY.merge(area[['TOWNNAME','AREA']], on=['TOWNNAME'], how = 'left')
        COUNTY = COUNTY.rename(columns={'AREA':'TOWN_AREA'})
        COUNTY['AVG_CBC'] = COUNTY['CBC']/COUNTY['TOTAL_AREA']
        COUNTY['AVG_point'] = COUNTY['point_count']/COUNTY['TOWN_AREA']
        
        df_ = gpd.GeoDataFrame(df[df['limit'] < 99], geometry=gpd.points_from_xy(df[df['limit'] < 99]['Longitude'], df[df['limit'] < 99]['Latitude']), crs="EPSG:4326") 
        df_ = df_.to_crs(epsg=3826) 
        df_
        
        # 3. 設定搜尋半徑 (例如：半徑 5公里 = 5000公尺)
        radius = np.arange(500,10001,100)
        for j in radius:
            # 4. 建立緩衝區 (畫圓)
            # 這會產生一堆圓形的幾何形狀
            buffers = df_.copy()
            buffers['geometry'] = buffers.buffer(j)
            
            # 5. 空間連結 (Spatial Join)
            # 邏輯：找出「原本的點」落在「哪些圓」裡面
            joined = gpd.sjoin(df_, buffers, how="inner", predicate="within")
            
            # 6. 計算數量
            # index_right 代表是「哪一個圓(測站)」
            # 我們計算每個圓裡面有多少個點
            counts = joined.groupby('index_right').size()
            
            # 7. 扣掉自己 (題目是問"其他"測站，通常不包含自己)
            df['Station_Count'] = counts
            
            # 8. (選用) 填補 NaN (有些測站附近完全沒人，會變 NaN，補 0)
            df['Station_Count'] = df['Station_Count'].fillna(0)
            COUNTY_count = df[df['CityName'] == k].copy()
            try:os.mkdir(f'./{CITY_ENG[k]}_count/{i}')
            except:i=i
            COUNTY_count.to_csv(f'./{CITY_ENG[k]}_count/{i}/{CITY_ENG[k]}_count_{j}m.csv', index=False)
print("Done")

Done


## 合併車禍資料

In [4]:
file = glob.glob('./accident/114/*/NPA_TMA2_*.csv')
total = pd.read_csv(file[0], sep= ',', skipinitialspace=True)
for i in range(1,len(file)):
    df_tmp = pd.read_csv(file[0], sep= ',', skipinitialspace=True)
    total = pd.concat([total,df_tmp])
total

C:\Users\nianz\AppData\Local\Temp\ipykernel_16508\2044588106.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  total = pd.read_csv(file[0], sep= ',', skipinitialspace=True)
C:\Users\nianz\AppData\Local\Temp\ipykernel_16508\2044588106.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tmp = pd.read_csv(file[0], sep= ',', skipinitialspace=True)
C:\Users\nianz\AppData\Local\Temp\ipykernel_16508\2044588106.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tmp = pd.read_csv(file[0], sep= ',', skipinitialspace=True)
C:\Users\nianz\AppData\Local\Temp\ipykernel_16508\2044588106.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tmp = pd.read_csv(file[0], sep= ',', skipinitialspace=True)
C:\Users\nianz\AppData\Local\Temp\ipykernel_16508\2044588106.py:4: DtypeW

,發生年度,發生月份,發生日期,發生時間,事故類別名稱,處理單位名稱警局層,發生地點,天候名稱,光線名稱,道路類別-第1當事者-名稱,...,當事者行動狀態子類別名稱,車輛撞擊部位大類別名稱-最初,車輛撞擊部位子類別名稱-最初,車輛撞擊部位大類別名稱-其他,車輛撞擊部位子類別名稱-其他,肇因研判大類別名稱-個別,肇因研判子類別名稱-個別,肇事逃逸類別名稱-是否肇逃,經度,緯度
0,2025,1.0,20250101.0,0.0,A2,基隆市警察局,基隆市安樂區安樂路一段95號前0.0公尺,晴,有照明且開啟,市區道路,...,向前直行中,機車與自行車,前車頭,NaN,NaN,駕駛者,觀看其他事故、活動、道路環境或車外資訊分心駕駛,否,121.733651,25.128823
1,2025,1.0,20250101.0,0.0,A2,基隆市警察局,基隆市安樂區安樂路一段95號前0.0公尺,晴,有照明且開啟,市區道路,...,NaN,NaN,NaN,NaN,NaN,無(非車輛駕駛人因素),物品(件)滾(滑行)或飛(掉)落,否,121.733651,25.128823
2,2025,1.0,20250101.0,400.0,A2,桃園市政府警察局,桃園市八德區長興一路附近 / 桃園市八德區長興二路附近,晴,有照明且開啟,市區道路,...,奔跑,其他,非汽、機及自行車,NaN,NaN,非駕駛者,穿越道路未注意左右來車,否,121.280105,24.933127
3,2025,1.0,20250101.0,400.0,A2,桃園市政府警察局,桃園市八德區長興一路附近 / 桃園市八德區長興二路附近,晴,有照明且開啟,市區道路,...,向前直行中,機車與自行車,前車頭,NaN,NaN,無(車輛駕駛者因素),尚未發現肇事因素,否,121.280105,24.933127
4,2025,1.0,20250101.0,400.0,A2,桃園市政府警察局,桃園市八德區長興一路附近 / 桃園市八德區長興二路附近,晴,有照明且開啟,市區道路,...,其他,其他,非汽、機及自行車,NaN,NaN,無(非車輛駕駛人因素),尚未發現肇事因素,否,121.280105,24.933127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81563,2025,1.0,20250131.0,235500.0,A2,高雄市政府警察局,高雄市楠梓區惠心街 / 高雄市楠梓區青埔街,雨,有照明且開啟,市區道路,...,向前直行中,機車與自行車,前車頭,NaN,NaN,駕駛者,未保持行車安全距離,否,120.322294,22.729380
81564,2025,1.0,20250131.0,235800.0,A2,高雄市政府警察局,高雄市楠梓區德民路 / 高雄市楠梓區德賢路,雨,有照明且開啟,市區道路,...,右轉彎,汽車,左前車頭(身),NaN,NaN,駕駛者,車輛未依規定暫停讓行人先行,否,120.304672,22.726881
81565,2025,1.0,20250131.0,235800.0,A2,高雄市政府警察局,高雄市楠梓區德民路 / 高雄市楠梓區德賢路,雨,有照明且開啟,市區道路,...,步行,其他,非汽、機及自行車,NaN,NaN,無(非車輛駕駛人因素),尚未發現肇事因素,否,120.304672,22.726881
81566,資料提供日期：114年11月01日,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 篩選資料

In [5]:
total_filtered = total[['發生年度','發生月份','發生日期', '發生地點','經度','緯度']]
total_filtered.to_csv('./accident/Taipei_NewTaipei_accident.csv', index=False)